In [1]:
import json
import pandas as pd
from bids import BIDSLayout

# ──────────────────────────────────────────────────────────
# 1. LOADING A BIDS DATASET
# ──────────────────────────────────────────────────────────
# Point BIDSLayout at the root of your BIDS directory.
# It indexes every file and parses entities from filenames.

layout = BIDSLayout("/Users/paradisdabbadon/Documents/Projects02/python_neuro/ds007185")

print("=" * 60)
print("1. DATASET OVERVIEW")
print("=" * 60)
print(f"  Subjects:  {layout.get_subjects()}")
print(f"  Sessions:  {layout.get_sessions()}")
print(f"  Tasks:     {layout.get_tasks()}")
print(f"  Runs:      {layout.get_runs()}")
print(f"  Datatypes: {layout.get_datatypes()}")
print(f"  Total files indexed: {len(layout.get())}")


1. DATASET OVERVIEW
  Subjects:  ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019', '020', '021', '022', '023', '024', '025', '026', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061']
  Sessions:  ['01', '03']
  Tasks:     ['animal']
  Runs:      [01, 02, 03]
  Datatypes: ['anat', 'func']
  Total files indexed: 489


In [2]:
# ──────────────────────────────────────────────────────────
# 2. QUERYING FILES WITH .get()
# ──────────────────────────────────────────────────────────
# .get() is the workhorse — filter by any BIDS entity.

print("\n" + "=" * 60)
print("2. QUERYING FILES")
print("=" * 60)

# Get all BOLD images
bold_files = layout.get(suffix="bold", extension=".nii.gz")
print(f"\n  All BOLD images ({len(bold_files)}):")
for f in bold_files:
    print(f"    {f.relpath}")

# Get event files only
event_files = layout.get(suffix="events", extension=".tsv")
print(f"\n  All event files ({len(event_files)}):")
for f in event_files:
    print(f"    {f.relpath}")

# Filter by run
run1_files = layout.get(run="01")
print(f"\n  All files for run-01 ({len(run1_files)}):")
for f in run1_files:
    print(f"    {f.relpath}")

# Combine filters: BOLD JSON sidecar for run-02
sidecar = layout.get(suffix="bold", extension=".json", run="02")
print(f"\n  BOLD sidecar for run-02:")
print(f"    {sidecar[0].relpath}")



2. QUERYING FILES

  All BOLD images (121):
    sub-001/ses-03/func/sub-001_ses-03_task-animal_run-01_bold.nii.gz
    sub-001/ses-03/func/sub-001_ses-03_task-animal_run-02_bold.nii.gz
    sub-002/ses-03/func/sub-002_ses-03_task-animal_run-01_bold.nii.gz
    sub-002/ses-03/func/sub-002_ses-03_task-animal_run-02_bold.nii.gz
    sub-003/ses-01/func/sub-003_ses-01_task-animal_run-01_bold.nii.gz
    sub-003/ses-01/func/sub-003_ses-01_task-animal_run-02_bold.nii.gz
    sub-004/ses-01/func/sub-004_ses-01_task-animal_run-01_bold.nii.gz
    sub-004/ses-01/func/sub-004_ses-01_task-animal_run-02_bold.nii.gz
    sub-005/ses-01/func/sub-005_ses-01_task-animal_run-01_bold.nii.gz
    sub-005/ses-01/func/sub-005_ses-01_task-animal_run-02_bold.nii.gz
    sub-006/ses-01/func/sub-006_ses-01_task-animal_run-01_bold.nii.gz
    sub-006/ses-01/func/sub-006_ses-01_task-animal_run-02_bold.nii.gz
    sub-007/ses-01/func/sub-007_ses-01_task-animal_run-01_bold.nii.gz
    sub-007/ses-01/func/sub-007_ses-01_task-a

In [3]:
# ──────────────────────────────────────────────────────────
# 3. READING FILE CONTENTS DIRECTLY
# ──────────────────────────────────────────────────────────
# BIDSLayout knows how to read TSVs (→ DataFrame) and
# JSONs (→ dict) natively via .get_...() helpers.

print("\n" + "=" * 60)
print("3. READING CONTENTS")
print("=" * 60)

# --- Event files → pandas DataFrames ---
events_run1 = layout.get(suffix="events", run="01")[0]
df = events_run1.get_df()       # Returns a pandas DataFrame
print(f"\n  Events for run-01 (shape: {df.shape}):")
print(df.to_string(index=False))

# --- JSON metadata ---
meta = layout.get(suffix="bold", extension=".json", run="01")[0]
metadata = meta.get_dict()      # Returns a plain dict
print(f"\n  Key acquisition parameters (run-01):")
print(f"    TR (RepetitionTime):    {metadata['RepetitionTime']} s")
print(f"    TE (EchoTime):          {metadata['EchoTime']} s")
print(f"    Flip angle:             {metadata['FlipAngle']}°")
print(f"    Multiband factor:       {metadata['MultibandAccelerationFactor']}")
print(f"    Slice thickness:        {metadata['SliceThickness']} mm")
print(f"    Number of slices:       {len(metadata['SliceTiming'])}")
print(f"    Phase encoding dir:     {metadata['PhaseEncodingDirection']}")


# ──────────────────────────────────────────────────────────
# 4. ACCESSING METADATA VIA get_metadata()
# ──────────────────────────────────────────────────────────
# For BOLD images, you can pull the associated JSON sidecar
# metadata without finding the JSON file manually. PyBIDS
# follows BIDS inheritance rules automatically.

print("\n" + "=" * 60)
print("4. AUTOMATIC METADATA ASSOCIATION")
print("=" * 60)

bold_run2 = layout.get(suffix="bold", extension=".nii.gz", run="02")[0]
meta2 = bold_run2.get_metadata()   # Finds the matching JSON sidecar
print(f"\n  Metadata for {bold_run2.filename}:")
print(f"    TR:          {meta2['RepetitionTime']} s")
print(f"    Task:        {meta2['TaskName']}")
print(f"    Scanner:     {meta2['Manufacturer']} {meta2['ManufacturersModelName']}")
print(f"    Field (T):   {meta2['MagneticFieldStrength']}")



3. READING CONTENTS

  Events for run-01 (shape: (14, 3)):
     onset  duration  trial_type
 84.964657        12           1
100.927745        12           2
 34.510309        12           3
 68.460517        12           3
182.428166        12           1
 51.461312        12           3
198.482344        12           2
133.543705        12           1
214.893955        12           1
117.282025        12           2
 17.809881        12           1
166.532088        12           2
149.461657        12           2
  1.141677        12           2

  Key acquisition parameters (run-01):
    TR (RepetitionTime):    2 s
    TE (EchoTime):          0.0302 s
    Flip angle:             90°
    Multiband factor:       3
    Slice thickness:        2 mm
    Number of slices:       75
    Phase encoding dir:     j-

4. AUTOMATIC METADATA ASSOCIATION

  Metadata for sub-001_ses-03_task-animal_run-02_bold.nii.gz:
    TR:          2 s
    Task:        animal
    Scanner:     Siemens Prisma
    

In [4]:
# ──────────────────────────────────────────────────────────
# 5. ENTITY PARSING — EXTRACTING INFO FROM FILENAMES
# ──────────────────────────────────────────────────────────
# Every BIDSFile object has .entities — a dict of parsed
# BIDS key-value pairs from the filename.

print("\n" + "=" * 60)
print("5. ENTITY PARSING")
print("=" * 60)

for f in layout.get(suffix="bold", extension=".nii.gz"):
    ent = f.entities
    print(f"\n  File: {f.filename}")
    print(f"    subject:   {ent.get('subject')}")
    print(f"    session:   {ent.get('session')}")
    print(f"    task:      {ent.get('task')}")
    print(f"    run:       {ent.get('run')}")
    print(f"    suffix:    {ent.get('suffix')}")
    print(f"    datatype:  {ent.get('datatype')}")



5. ENTITY PARSING

  File: sub-001_ses-03_task-animal_run-01_bold.nii.gz
    subject:   001
    session:   03
    task:      animal
    run:       01
    suffix:    bold
    datatype:  func

  File: sub-001_ses-03_task-animal_run-02_bold.nii.gz
    subject:   001
    session:   03
    task:      animal
    run:       02
    suffix:    bold
    datatype:  func

  File: sub-002_ses-03_task-animal_run-01_bold.nii.gz
    subject:   002
    session:   03
    task:      animal
    run:       01
    suffix:    bold
    datatype:  func

  File: sub-002_ses-03_task-animal_run-02_bold.nii.gz
    subject:   002
    session:   03
    task:      animal
    run:       02
    suffix:    bold
    datatype:  func

  File: sub-003_ses-01_task-animal_run-01_bold.nii.gz
    subject:   003
    session:   01
    task:      animal
    run:       01
    suffix:    bold
    datatype:  func

  File: sub-003_ses-01_task-animal_run-02_bold.nii.gz
    subject:   003
    session:   01
    task:      animal
    run

In [5]:
# ──────────────────────────────────────────────────────────
# 6. PRACTICAL EXAMPLE — BUILDING A FIRST-LEVEL MODEL SPEC
# ──────────────────────────────────────────────────────────
# Loop through all runs, pair each BOLD with its events and
# TR — exactly what you'd feed into nilearn or nipype.

print("\n" + "=" * 60)
print("6. BUILDING A FIRST-LEVEL ANALYSIS SPEC")
print("=" * 60)

bold_files = layout.get(
    subject="061",
    task="animal",
    suffix="bold",
    extension=".nii.gz",
    return_type="filename"
)

for bold_path in sorted(bold_files):
    # Get the matching BIDSFile object to use its helpers
    bf = layout.get(suffix="bold", extension=".nii.gz",
                    return_type="object",
                    run=layout.parse_file_entities(bold_path)["run"])[0]

    meta = bf.get_metadata()
    tr = meta["RepetitionTime"]

    # Find the associated events file
    entities = bf.entities
    events = layout.get(
        subject=entities["subject"],
        session=entities["session"],
        task=entities["task"],
        run=entities["run"],
        suffix="events",
        extension=".tsv"
    )

    if events:
        ev_df = events[0].get_df()
        conditions = sorted(ev_df["trial_type"].unique())

        print(f"\n  Run {entities['run']}:")
        print(f"    BOLD:       {bf.filename}")
        print(f"    Events:     {events[0].filename}")
        print(f"    TR:         {tr} s")
        print(f"    Conditions: {conditions}")
        print(f"    Trials:     {len(ev_df)}")

        # Per-condition summary
        for cond in conditions:
            subset = ev_df[ev_df["trial_type"] == cond]
            print(f"      Type {cond}: {len(subset)} trials, "
                  f"onsets at {sorted(round(x, 1) for x in subset['onset'])} s")

print("\n" + "=" * 60)
print("Done! These building blocks scale to hundreds of subjects.")
print("=" * 60)



6. BUILDING A FIRST-LEVEL ANALYSIS SPEC

  Run 01:
    BOLD:       sub-001_ses-03_task-animal_run-01_bold.nii.gz
    Events:     sub-001_ses-03_task-animal_run-01_events.tsv
    TR:         2 s
    Conditions: [np.int64(1), np.int64(2), np.int64(3)]
    Trials:     14
      Type 1: 5 trials, onsets at [17.8, 85.0, 133.5, 182.4, 214.9] s
      Type 2: 6 trials, onsets at [1.1, 100.9, 117.3, 149.5, 166.5, 198.5] s
      Type 3: 3 trials, onsets at [34.5, 51.5, 68.5] s

  Run 02:
    BOLD:       sub-001_ses-03_task-animal_run-02_bold.nii.gz
    Events:     sub-001_ses-03_task-animal_run-02_events.tsv
    TR:         2 s
    Conditions: [np.int64(1), np.int64(2), np.int64(3)]
    Trials:     13
      Type 1: 4 trials, onsets at [17.5, 99.5, 132.2, 164.3] s
      Type 2: 3 trials, onsets at [115.8, 148.5, 181.1] s
      Type 3: 6 trials, onsets at [1.3, 34.5, 50.3, 67.2, 83.3, 197.2] s

Done! These building blocks scale to hundreds of subjects.
